# Exercise 1

## Part 1

Loading finnish and multilingual tokenizer to check vocab_size:

In [1]:
from transformers import AutoTokenizer

fintokenizer = AutoTokenizer.from_pretrained("TurkuNLP/bert-base-finnish-cased-v1")
multitokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased")

In [2]:
print(fintokenizer.vocab_size)
print(multitokenizer.vocab_size)

50105
119547


<div class="alert alert-block alert-success">
<b>a)</b> Finnish tokenizer's vocab_size is 50105 tokens, and multilingual tokenizer has just under 200 000 tokens. The multilingual tokenizer though holds tokens from 104 different languages, and is one of the largest. Though, I suspect the token amount per language to be smaller, if every language has the same amount tokens -> (119547/104=1149,49...), only around 1150 tokens are listed to one language. This would mean, that the specified finnish BERT has a better chance to work than the multilangual in finnish language.
</div>

In [3]:
fintokenized: list = []
counter: int = 0

with open("kissa.txt") as f:
    for line in f:
        for word in line.split():
            fintokenized.extend(fintokenizer.tokenize(word))
            counter += 1

print(counter)

print(fintokenized)
print(len(fintokenized))

multitokenized: list = []

with open("kissa.txt") as f:
    for line in f:
        for word in line.split():
            multitokenized.extend(multitokenizer.tokenize(word))

print(multitokenized)
print(len(multitokenized))


145
['Kissa', 'eli', 'kes', '##yk', '##issa', 'tai', 'kotik', '##issa', '(', 'Feli', '##s', 'ca', '##tus', ',', '[', '1', ']', '[', '2', ']', 'aiemmin', 'Feli', '##s', 'sil', '##vest', '##ris', 'ca', '##tus', ')', 'on', 'afr', '##ikan', '##vill', '##ikissa', '##sta', '(', 'Feli', '##s', 'ly', '##bi', '##ca', ')', 'polv', '##e', '##utuva', 'ja', 'peto', '##eläinten', '(', 'Car', '##ni', '##vo', '##ra', ')', 'lahkon', 'kissa', '##eläinten', '(', 'Feli', '##da', '##e', ')', 'heimo', '##on', 'kuuluva', 'kes', '##y', 'nisä', '##käs', '##laji', '.', 'Kissa', '##t', 'ovat', 'suosittuja', 'lemmikki', '##eläimiä', ',', 'ja', 'etenkin', 'maaseudulla', 'ne', 'ovat', 'aina', 'olleet', 'hyödyllisiä', 'hiir', '##ten', 'ja', 'muiden', 'tuho', '##laisten', 'pyyd', '##ystä', '##jinä', '.', 'Ihminen', 'alkoi', 'pitää', 'villi', '##kis', '##soja', 'vilja', '##varasto', '##jen', 'suojeli', '##joina', 'Lähi', '-', 'idässä', 'pian', 'maanviljel', '##yksen', 'keksi', '##misen', 'jälkeen', 'yli', '10', '000',

<div class="alert alert-block alert-success">
<b>b)</b> The Finnish tokenizer tokenizes each word to around 258 tokens when the text file has 145 words. The multilingual tokenizer however tokenized the 145 words to 387 tokens. This was expected, since the multilingual tokenizer tokenizes the words to unnecessary subwords (for example kissa => 'Kiss', '##a') 
</div>

<div class="alert alert-block alert-success">
<b>c)</b> With the openAI GPT-5.x tokenizer visualizer, the same text is tokenized to 392 tokens and with GPT-4 the text is tokenized to 506 tokens. OpenAI uses multilingual models, so a similar type of tokenization results as above was expected, though I expected it to be a tad bit better than the bert tokenizer.
</div>

## Part 2

Loading the dataset by using the dataset python library:

In [4]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

### Printing the dataset to see data structure

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


Using pipeline this time:

In [6]:
from transformers import pipeline

MODEL_NAME = 'TurkuNLP/bert-base-finnish-cased-v1'

pipe = pipeline('text-generation', model=MODEL_NAME)

tokenizer = pipe.tokenizer

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertLMHeadModel LOAD REPORT from: TurkuNLP/bert-base-finnish-cased-v1
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
print(dataset["train"]["text"][:5])

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

In [8]:
def tokenize(example):
    return tokenizer(
        example["text"],
        max_length=128,
        truncation=True,
    )

In [11]:
def tokenize_dataset(example: dict):
    tokenized = tokenizer(example["text"])
    example["text"] = tokenized
    return example

In [12]:
returned_dataset = dataset.map(tokenize_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (609 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [9]:
returned_dataset2 = dataset.map(tokenize)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [11]:
print(returned_dataset2["train"][:2])

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [13]:
print(returned_dataset["train"]["text"][:2])

[{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

<div class="alert alert-block alert-success">
I can't figure out how to get each token's vector as one dict so that the returned_dataset would be a list of dictionaries; because I assume that the above code adds all the vectors to one list.
</div>